# STAGE 5 — SCENARIO SIMULATION ENGINE EXPLORATION

This notebook demonstrates the **Scenario Simulation Engine** of Spendable.

### Core Question Answered:
> *"What happens to my financial runway and safe spendable capacity if I change something?"*

### Supported Scenario Types Demonstrated:
1. **ONE_TIME_EXPENSE**: *"What if I spend ৳3,000 today?"*
2. **ADDITIONAL_INCOME**: *"What if I receive an extra ৳5,000?"*
3. **ADDITIONAL_COMMITMENT**: *"What if I take on an additional ৳4,000 monthly obligation?"*
4. **SPENDING_REDUCTION**: *"What if I reduce my discretionary spending by 20%?"*
5. **INCOME_DELAY**: *"What if my expected income is delayed beyond 30 days?"*
6. **EXPENSE EXCEEDING BALANCE**: *"What if I attempt to spend more than my current balance?"*

### Key Design Principle:
- **Zero Base Mutation**: Base financial state remains untouched; all transformations operate on derived scenario states.
- **Spendable Engine Reuse**: Recalculates underlying balance, trajectory, and commitments, passing them through the authoritative `SpendableCalculator`.


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup sys.path to backend
sys.path.insert(0, str(Path.cwd() / "backend"))

from app.engine.schema import SpendableOutput, LiquidityState
from app.engine.calculator import SpendableCalculator
from app.scenario.schema import ScenarioInput, ScenarioResult, ScenarioType
from app.scenario.simulator import ScenarioSimulator
from app.forecasting.models import CashFlowForecastModel

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)


## 1. Baseline Financial State Preparation

We load a sample user snapshot from the test split and calculate their authoritative baseline Spendable capacity.


In [ ]:
data_path = Path.cwd() / "data" / "features"
test_df = pd.read_csv(data_path / "features_test.csv")
train_df = pd.read_csv(data_path / "features_train.csv")

# Train forecasting model for trajectory predictions
model = CashFlowForecastModel()
model.fit(train_df)

sample_row = test_df.iloc[0].to_dict()
user_id = str(sample_row.get("account_id", "USER_001"))
snapshot_time = str(sample_row.get("snapshot_time", "2026-03-01"))
current_balance = float(sample_row.get("current_balance", 0.0))

forecast = model.predict_snapshot(sample_row)

simulator = ScenarioSimulator()
base_res = simulator.simulate(
    user_id=user_id,
    snapshot_time=snapshot_time,
    current_balance=current_balance,
    features=sample_row,
    commitments=[],
    forecast=forecast,
    scenario_input=None
)

print("--- Authoritative Base Financial State ---")
print(f"User ID: {base_res.user_id}")
print(f"Current Balance: ৳{base_res.base_current_balance:,.2f}")
print(f"Base Spendable Amount: ৳{base_res.base_spendable_amount:,.2f}")
print(f"Forecasted Min Balance: ৳{base_res.base_forecasted_minimum_balance:,.2f}")
print(f"Safety Reserve: ৳{base_res.base_safety_reserve:,.2f}")
print(f"Liquidity State: {base_res.base_liquidity_state.value}")


--- Authoritative Base Financial State ---
User ID: ACC-0497
Current Balance: ৳59,835.05
Base Spendable Amount: ৳22,552.76
Forecasted Min Balance: ৳34,284.50
Safety Reserve: ৳11,731.74
Liquidity State: HEALTHY


## 2. Hypothetical Scenario Executions

We run 6 representative scenarios and inspect their impact on spendable capacity, safety reserve, minimum balance, and liquidity state.


In [ ]:
scenarios_list = [
    ScenarioInput(scenario_type=ScenarioType.ONE_TIME_EXPENSE, amount=3000.0, description="Spend ৳3,000 today"),
    ScenarioInput(scenario_type=ScenarioType.ADDITIONAL_INCOME, amount=5000.0, description="Receive ৳5,000 extra income"),
    ScenarioInput(scenario_type=ScenarioType.ADDITIONAL_COMMITMENT, amount=4000.0, description="Add ৳4,000 recurring obligation"),
    ScenarioInput(scenario_type=ScenarioType.SPENDING_REDUCTION, percentage=20.0, description="Reduce spending by 20%"),
    ScenarioInput(scenario_type=ScenarioType.INCOME_DELAY, amount=10000.0, description="Delay ৳10,000 income"),
    ScenarioInput(scenario_type=ScenarioType.ONE_TIME_EXPENSE, amount=current_balance + 20000.0, description="Spend ৳(Balance + 20k) [Over-balance]"),
]

scenario_results = []

for sc_inp in scenarios_list:
    res = simulator.simulate(
        user_id=user_id,
        snapshot_time=snapshot_time,
        current_balance=current_balance,
        features=sample_row,
        commitments=[],
        forecast=forecast,
        scenario_input=sc_inp
    )
    scenario_results.append(res)
    print(f"Executed: [{res.scenario_type.value}] -> {res.scenario_description}")
    print(f"  Spendable: ৳{res.base_spendable_amount:,.2f} -> ৳{res.scenario_spendable_amount:,.2f} (Delta: ৳{res.spendable_delta:+,.2f})")
    print(f"  Liquidity State: {res.base_liquidity_state.value} -> {res.scenario_liquidity_state.value}\n")


Executed: [ONE_TIME_EXPENSE] -> Spend ৳3,000 today
  Spendable: ৳22,552.76 -> ৳19,552.76 (Delta: ৳-3,000.00)
  Liquidity State: HEALTHY -> HEALTHY

Executed: [ADDITIONAL_INCOME] -> Receive ৳5,000 extra income
  Spendable: ৳22,552.76 -> ৳27,552.76 (Delta: ৳+5,000.00)
  Liquidity State: HEALTHY -> HEALTHY

Executed: [ADDITIONAL_COMMITMENT] -> Add ৳4,000 recurring obligation
  Spendable: ৳22,552.76 -> ৳22,552.76 (Delta: ৳+0.00)
  Liquidity State: HEALTHY -> HEALTHY

Executed: [SPENDING_REDUCTION] -> Reduce spending by 20%
  Spendable: ৳22,552.76 -> ৳29,665.66 (Delta: ৳+7,112.90)
  Liquidity State: HEALTHY -> HEALTHY

Executed: [INCOME_DELAY] -> Delay ৳10,000 income
  Spendable: ৳22,552.76 -> ৳12,552.76 (Delta: ৳-10,000.00)
  Liquidity State: HEALTHY -> WATCH

Executed: [ONE_TIME_EXPENSE] -> Spend ৳(Balance + 20k) [Over-balance]
  Spendable: ৳22,552.76 -> ৳0.00 (Delta: ৳-22,552.76)
  Liquidity State: HEALTHY -> PRESSURED



## 3. Scenario Comparison Matrix

Detailed side-by-side comparison of baseline vs. simulated scenarios.


In [ ]:
table_data = []

# Base row
table_data.append({
    "Scenario": "BASELINE (Original State)",
    "Balance [BDT]": f"৳{base_res.base_current_balance:,.2f}",
    "Base Spendable": f"৳{base_res.base_spendable_amount:,.2f}",
    "Scenario Spendable": f"৳{base_res.base_spendable_amount:,.2f}",
    "Spendable Delta": "৳0.00",
    "Forecasted Min Bal": f"৳{base_res.base_forecasted_minimum_balance:,.2f}",
    "Safety Reserve": f"৳{base_res.base_safety_reserve:,.2f}",
    "Liquidity State": base_res.base_liquidity_state.value
})

for r in scenario_results:
    table_data.append({
        "Scenario": r.scenario_description,
        "Balance [BDT]": f"৳{r.scenario_current_balance:,.2f}",
        "Base Spendable": f"৳{r.base_spendable_amount:,.2f}",
        "Scenario Spendable": f"৳{r.scenario_spendable_amount:,.2f}",
        "Spendable Delta": f"৳{r.spendable_delta:+,.2f}",
        "Forecasted Min Bal": f"৳{r.scenario_forecasted_minimum_balance:,.2f}",
        "Safety Reserve": f"৳{r.scenario_safety_reserve:,.2f}",
        "Liquidity State": r.scenario_liquidity_state.value
    })

df_matrix = pd.DataFrame(table_data)
print("--- SCENARIO COMPARISON MATRIX ---")
print(df_matrix.to_string(index=False))


--- SCENARIO COMPARISON MATRIX ---
                             Scenario Balance [BDT] Base Spendable Scenario Spendable Spendable Delta Forecasted Min Bal Safety Reserve Liquidity State
            BASELINE (Original State)    ৳59,835.05     ৳22,552.76         ৳22,552.76           ৳0.00         ৳34,284.50     ৳11,731.74         HEALTHY
                   Spend ৳3,000 today    ৳56,835.05     ৳22,552.76         ৳19,552.76      ৳-3,000.00         ৳31,284.50     ৳11,731.74         HEALTHY
          Receive ৳5,000 extra income    ৳64,835.05     ৳22,552.76         ৳27,552.76      ৳+5,000.00         ৳39,284.50     ৳11,731.74         HEALTHY
      Add ৳4,000 recurring obligation    ৳59,835.05     ৳22,552.76         ৳22,552.76          ৳+0.00         ৳34,284.50     ৳11,731.74         HEALTHY
               Reduce spending by 20%    ৳59,835.05     ৳22,552.76         ৳29,665.66      ৳+7,112.90         ৳39,613.93      ৳9,948.27         HEALTHY
                 Delay ৳10,000 income    ৳59,835.05  

## 4. Visualizing Spendable Capacity Deltas Across Scenarios


In [ ]:
names = [r.scenario_description for r in scenario_results]
deltas = [r.spendable_delta for r in scenario_results]
colors = ["crimson" if d < 0 else "teal" for d in deltas]

plt.figure(figsize=(12, 5))
bars = plt.barh(names, deltas, color=colors, edgecolor="black", alpha=0.85)
plt.axvline(0, color="gray", linestyle="--", linewidth=1.5)
plt.title("Impact of Financial Scenarios on Safe Spendable Capacity [BDT]")
plt.xlabel("Spendable Delta [BDT]")

for bar in bars:
    width = bar.get_width()
    offset = 500 if width >= 0 else -1500
    plt.text(width + offset, bar.get_y() + bar.get_height()/2, f"৳{width:+,.0f}", va="center", fontweight="bold")

plt.tight_layout()
plt.show()
